# CSV Generation and Build Manifest

This notebook generates the dataset build manifest CSV from an existing `final_split_dataset` directory.


In [ ]:
import os
import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
print(f'Running in Colab: {IN_COLAB}')
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

BASE_DIR = Path(os.environ.get('NAILSCAN_BASE_DIR', './modular2_all_in_one')).resolve()
print(f'BASE_DIR: {BASE_DIR}')


In [ ]:
# --- Per-class balanced targets ---
# Strategy: Pool all originals, split 70/20/10, then augment WITHIN each split.
TRAIN_TARGET_PER_CLASS = 5250
VAL_TARGET_PER_CLASS   = 1500
TEST_TARGET_PER_CLASS  = 750
# Total per class: 7,500 | Grand total: 60,000 images (7,500 × 8 classes)
print(f"Targets per class → Train: {TRAIN_TARGET_PER_CLASS}, Val: {VAL_TARGET_PER_CLASS}, Test: {TEST_TARGET_PER_CLASS}")
print(f"Grand total per class: {TRAIN_TARGET_PER_CLASS + VAL_TARGET_PER_CLASS + TEST_TARGET_PER_CLASS}")
print(f"Grand total all classes: {(TRAIN_TARGET_PER_CLASS + VAL_TARGET_PER_CLASS + TEST_TARGET_PER_CLASS) * 8}")

DEFAULT_TARGET_CLASSES = [
    'acral_lentiginous_melanoma',
    'beau_lines',
    'blue_finger',
    'clubbing',
    'healthy_nails',
    'koilonychia',
    'muehrckes_lines',
    'pitting',
]

TARGETS = {
    'train': TRAIN_TARGET_PER_CLASS,
    'val': VAL_TARGET_PER_CLASS,
    'test': TEST_TARGET_PER_CLASS,
}


In [ ]:
import pandas as pd

SPLIT_OUTPUT = BASE_DIR / 'final_split_dataset'
SPLITS = ['train', 'val', 'test']

if not SPLIT_OUTPUT.exists():
    raise FileNotFoundError(f'Expected split dataset at {SPLIT_OUTPUT}.')

discovered_classes = set()
for split in SPLITS:
    split_dir = SPLIT_OUTPUT / split
    if split_dir.exists():
        discovered_classes.update([p.name for p in split_dir.iterdir() if p.is_dir()])

TARGET_CLASSES = sorted(discovered_classes) if discovered_classes else DEFAULT_TARGET_CLASSES

build_rows = []
for split in SPLITS:
    for cls in TARGET_CLASSES:
        class_dir = SPLIT_OUTPUT / split / cls
        if not class_dir.exists():
            orig_count = aug_count = total_count = 0
        else:
            files = [p for p in class_dir.iterdir() if p.is_file()]
            orig_count = sum(p.name.startswith('orig_') for p in files)
            aug_count = sum(p.name.startswith('aug_') for p in files)
            total_count = len(files)
            other_count = total_count - orig_count - aug_count
            if other_count:
                print(f'Note: {other_count} unclassified files in {class_dir}.')

        build_rows.append({
            'class': cls,
            'split': split,
            'original_images': orig_count,
            'augmented_added': total_count - orig_count,
            'final_count': total_count,
            'target': TARGETS.get(split),
        })

build_df = pd.DataFrame(build_rows)
build_csv_path = BASE_DIR / 'dataset_build_manifest.csv'
build_df.to_csv(build_csv_path, index=False)
print(build_df.to_string(index=False))
print(f'\nManifest saved to: {build_csv_path}')
